# INTERACTIVE SQL PLAYBOOK (33 SCENARIOS)

This Google Colab notebook provides an interactive masterclass for learning SQL, covering 33 scenarios from foundational concepts to advanced Google/FAANG-level challenges. It features an in-memory SQLite database, pre-seeded with realistic coffee chain transaction data, and a Flask-based web interface to explore each SQL scenario step-by-step.

## Features

- **33 Interactive SQL Scenarios**: Ranging from basic selections and filters to complex window functions, recursive CTEs, JSON parsing, and advanced analytical problems like Gaps & Islands, Sessionization, Pareto Principle, and MoM retention.
- **In-Memory SQLite Database**: A self-contained, ephemeral database is set up with tables for `stores`, `customers`, `products`, `orders`, `order_items`, `employees`, `store_logs`, and `store_shifts`.
- **Realistic Sample Data**: The database is populated with data that mimics a coffee chain's operations.
- **Step-by-Step Execution**: Each scenario is broken down into logical steps, showing the SQL query for each step and its resulting output.
- **Flask Web Interface**: A simple web application built with Flask, HTML, CSS, and vanilla JavaScript provides a user-friendly interface to navigate through scenarios and view query results directly in your browser or within Colab.
- **Colab Proxy Integration**: Seamless integration with Google Colab's proxy service allows access to the Flask app via a secure external link.

## How to Use

1.  **Open in Google Colab**: Click the "Open in Colab" badge (or similar link) to launch the notebook in your Google Colab environment.
2.  **Run All Cells**: Execute all code cells in the notebook sequentially. The notebook is designed to be self-contained, installing dependencies and setting up the environment automatically.
3.  **Access the Web Interface**: Once all cells have run, a URL will be printed at the end (typically starting with `https://...colab.dev` or `http://127.0.0.1:5000` for local environments). Click this link to open the INTERACTIVE SQL PLAYBOOK in your browser.
4.  **Explore Scenarios**: Use the dropdown menu or the "Previous" and "Next" buttons in the web interface to navigate through the 33 SQL scenarios. For each scenario, you can view the question, the steps involved, the SQL code for each step, and the live results from the in-memory database.

## Project Structure

-   **BLOCK 1: Dependency Installation & Workspace Hygiene**: Installs `Flask` and `Pandas`, and clears port 5000.
-   **BLOCK 2: In-Memory Relational Database Setup**: Defines and initializes the SQLite database schema and seeds it with sample data.
-   **BLOCK 3: Curriculum Catalog**: Contains a Python list of dictionaries (`SCENARIOS`), where each dictionary describes an SQL scenario, its question, and the step-by-step SQL queries.
-   **BLOCK 4: Embedded Frontend Interface**: The HTML, CSS, and JavaScript for the single-page web application.
-   **BLOCK 5: Flask Route Handlers & Query Execution Engine**: Flask application logic to serve the HTML and execute SQL queries using `pandas.read_sql_query`.
-   **BLOCK 6: Background Thread Worker & Colab Proxy Bridge**: Starts the Flask application in a background thread and generates the Colab public URL.

## Technologies Used

-   **Python**
-   **Flask**: Web framework for serving the INTERACTIVE SQL PLAYBOOK.
-   **Pandas**: Used for executing SQL queries against the SQLite database and formatting results.
-   **SQLite**: Lightweight, in-memory relational database.
-   **HTML, CSS, JavaScript**: Frontend for the interactive masterclass.

## Contributing

Feel free to fork this repository, add new scenarios, improve existing explanations, or enhance the UI. Pull requests are welcome!

## License

This project is open-source and available under the MIT License.

In [1]:
# ==============================================================================
# ☕ INTERACTIVE SQL PLAYBOOK (ALL 33 SCENARIOS)
# Single-Cell Google Colab Self-Contained Runner
# ==============================================================================

# ==============================================================================
# BLOCK 1: DEPENDENCY INSTALLATION & WORKSPACE HYGIENE
# Purpose: Ensures Flask and Pandas are available in the container environment,
#          then frees TCP port 5000 from previous sessions to prevent Errno 98.
# ==============================================================================
import subprocess, time, sys

print("[1/5] Installing core dependencies (Flask, Pandas)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "pandas"])

print("[2/5] Clearing port 5000 of any lingering processes...")
subprocess.run(["fuser", "-k", "5000/tcp"], stderr=subprocess.DEVNULL)

[1/5] Installing core dependencies (Flask, Pandas)...
[2/5] Clearing port 5000 of any lingering processes...


CompletedProcess(args=['fuser', '-k', '5000/tcp'], returncode=1)

In [2]:
# ==============================================================================
# BLOCK 2: IN-MEMORY RELATIONAL DATABASE SETUP
# Purpose: Builds an in-memory SQLite schema in RAM and seeds realistic coffee
#          chain transaction data, employees, shifts, and nested JSON equipment logs.
# ==============================================================================
import sqlite3
import pandas as pd

def init_db():
    """
    Initializes an in-memory SQLite database instance.
    check_same_thread=False allows multiple concurrent request threads from Flask
    to safely read and execute queries on the same connection.
    """
    conn = sqlite3.connect(':memory:', check_same_thread=False)
    cursor = conn.cursor()
    cursor.executescript("""
    CREATE TABLE stores (
        store_id INTEGER PRIMARY KEY,
        city TEXT NOT NULL,
        store_name TEXT NOT NULL,
        opened_year INTEGER NOT NULL
    );

    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        loyalty_level TEXT,
        join_date TEXT NOT NULL
    );

    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL
    );

    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        store_id INTEGER,
        order_date TEXT NOT NULL,
        total_amount REAL NOT NULL
    );

    CREATE TABLE order_items (
        item_id INTEGER PRIMARY KEY,
        order_id INTEGER,
        product_id INTEGER,
        quantity INTEGER NOT NULL,
        item_price REAL NOT NULL
    );

    CREATE TABLE employees (
        employee_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        title TEXT NOT NULL,
        manager_id INTEGER
    );

    CREATE TABLE store_logs (
        log_id INTEGER PRIMARY KEY,
        store_id INTEGER,
        log_payload TEXT NOT NULL
    );

    CREATE TABLE store_shifts (
        shift_id INTEGER PRIMARY KEY,
        employee_id INTEGER,
        store_id INTEGER,
        start_time TEXT NOT NULL,
        end_time TEXT NOT NULL
    );

    INSERT INTO stores VALUES
    (1, 'Seattle', 'Downtown Flagship', 2018),
    (2, 'Portland', 'Pearl District', 2020),
    (3, 'Austin', 'South Congress', 2021),
    (4, 'Denver', 'LoDo Station', 2022);

    INSERT INTO customers VALUES
    (101, 'Alice Smith', 'Gold', '2023-01-15'),
    (102, 'Bob Jones', NULL, '2023-03-22'),
    (103, 'Charlie Brown', 'Silver', '2023-05-10'),
    (104, 'Diana Prince', 'Gold', '2023-06-01'),
    (105, 'Evan Wright', NULL, '2023-07-19'),
    (106, 'Fiona Gallagher', 'Platinum', '2023-08-11'),
    (107, 'George Clark', NULL, '2024-01-05'),
    (108, 'Hannah Abbott', 'Bronze', '2024-01-10');

    INSERT INTO products VALUES
    (1, 'Caramel Macchiato', 'Espresso', 5.25),
    (2, 'Oat Milk Latte', 'Espresso', 4.75),
    (3, 'Vanilla Latte', 'Espresso', 4.75),
    (4, 'Americano', 'Espresso', 3.50),
    (5, 'Drip Coffee', 'Brewed', 2.95),
    (6, 'Cold Brew', 'Brewed', 4.25),
    (7, 'Nitro Cold Brew', 'Brewed', 4.25),
    (8, 'Butter Croissant', 'Bakery', 3.50),
    (9, 'Almond Scone', 'Bakery', 3.50),
    (10, 'Chocolate Cookie', 'Bakery', 3.50),
    (11, 'Blueberry Muffin', 'Bakery', 3.75),
    (12, 'Avocado Toast', 'Bakery', 7.50);

    INSERT INTO orders VALUES
    (1001, 101, 1, '2024-02-01', 10.00),
    (1002, 102, 1, '2024-02-01', 4.75),
    (1003, 103, 2, '2024-02-02', 12.50),
    (1004, 101, 1, '2024-02-03', 15.25),
    (1005, 104, 3, '2024-02-03', 8.25),
    (1006, 105, 2, '2024-02-04', 3.50),
    (1007, 103, 3, '2024-02-05', 18.00),
    (1008, 106, 4, '2024-02-06', 22.50),
    (1009, 101, 1, '2024-02-07', 9.50),
    (1010, 101, 1, '2024-01-15', 14.00),
    (1011, 103, 2, '2024-01-18', 19.50),
    (1012, 102, 1, '2024-03-01', 8.00);

    INSERT INTO order_items VALUES
    (1, 1001, 1, 1, 5.25), (2, 1001, 2, 1, 4.75),
    (3, 1002, 2, 1, 4.75),
    (4, 1003, 1, 1, 5.25), (5, 1003, 12, 1, 7.50),
    (6, 1004, 1, 2, 10.50), (7, 1004, 11, 1, 3.75),
    (8, 1005, 6, 1, 4.25), (9, 1005, 8, 1, 3.50),
    (10, 1006, 8, 1, 3.50),
    (11, 1007, 12, 2, 15.00), (12, 1007, 5, 1, 2.95),
    (13, 1008, 1, 2, 10.50), (14, 1008, 12, 1, 7.50), (15, 1008, 6, 1, 4.25),
    (16, 1009, 2, 2, 9.50);

    INSERT INTO employees VALUES
    (1, 'Marcus Vance', 'CEO & Founder', NULL),
    (2, 'Sarah Jenkins', 'Regional Director', 1),
    (3, 'David Ross', 'Seattle Manager', 2),
    (4, 'Elena Rostova', 'Portland Manager', 2),
    (5, 'Tom Holland', 'Lead Barista', 3),
    (6, 'Zendaya Coleman', 'Senior Barista', 3);

    INSERT INTO store_logs VALUES
    (1, 1, '{"shift": "Morning", "manager": "Sarah", "equip_status": {"espresso_machine": "OK", "grinder": "CALIBRATED"}}'),
    (2, 2, '{"shift": "Afternoon", "manager": "Dave", "equip_status": {"espresso_machine": "SERVICE_REQUIRED", "grinder": "OK"}}'),
    (3, 3, '{"shift": "Morning", "manager": "Elena", "equip_status": {"espresso_machine": "OK", "grinder": "OK"}}');

    INSERT INTO store_shifts VALUES
    (1, 5, 1, '2024-02-01 07:00:00', '2024-02-01 13:00:00'),
    (2, 6, 1, '2024-02-01 08:30:00', '2024-02-01 15:30:00'),
    (3, 3, 1, '2024-02-01 08:00:00', '2024-02-01 16:00:00'),
    (4, 4, 2, '2024-02-01 09:00:00', '2024-02-01 17:00:00');
    """)
    conn.commit()
    return conn

# Bootstrap the shared database connection
db_conn = init_db()

In [3]:
# ==============================================================================
# BLOCK 3: CURRICULUM CATALOG (33 INTERACTIVE SQL SCENARIOS)
# Purpose: Master list of 33 technical drills spanning fundamentals, window
#          functions, pivoting, recursive CTEs, and senior/FAANG scenarios.
# ==============================================================================
SCENARIOS = [
    {
        "id": 0,
        "title": "Basic Selection, Filtering & Sorting",
        "question": "Which espresso items cost over $4.50, ordered highest to lowest?",
        "tables": ["products"],
        "steps": [
            {"step": 1, "label": "FROM", "keyword": "FROM products", "explanation": "Identifies target source table and generates initial evaluation set.", "sql": "SELECT name, category, price FROM products;"},
            {"step": 2, "label": "WHERE", "keyword": "WHERE category = 'Espresso' AND price > 4.50", "explanation": "Row predicate: Filters out items not matching 'Espresso' and $4.50 price threshold.", "sql": "SELECT name, category, price FROM products WHERE category = 'Espresso' AND price > 4.50;"},
            {"step": 3, "label": "ORDER BY", "keyword": "ORDER BY price DESC", "explanation": "Post-filtering ordering: Sorts result matrix by price attribute descending.", "sql": "SELECT name, category, price FROM products WHERE category = 'Espresso' AND price > 4.50 ORDER BY price DESC;"}
        ]
    },
    {
        "id": 1,
        "title": "NULL Imputation & Conditional Branching",
        "question": "How do we normalize names, replace missing loyalty tiers, and group VIP status?",
        "tables": ["customers"],
        "steps": [
            {"step": 1, "label": "UPPER()", "keyword": "UPPER(name)", "explanation": "Applies case transformation to customer names.", "sql": "SELECT name, UPPER(name) AS clean_name FROM customers;"},
            {"step": 2, "label": "COALESCE()", "keyword": "COALESCE(loyalty_level, 'Standard Tier')", "explanation": "Substitutes missing database NULL values with a default text literal.", "sql": "SELECT UPPER(name) AS clean_name, COALESCE(loyalty_level, 'Standard Tier') AS clean_loyalty FROM customers;"},
            {"step": 3, "label": "CASE WHEN", "keyword": "CASE WHEN loyalty_level IN ('Gold', 'Platinum') THEN 'VIP' ELSE 'Guest' END", "explanation": "Evaluates conditional branches sequentially to assign customer status categories.", "sql": "SELECT UPPER(name) AS clean_name, COALESCE(loyalty_level, 'Standard Tier') AS clean_loyalty, CASE WHEN loyalty_level IN ('Gold', 'Platinum') THEN 'VIP Member' ELSE 'Guest' END AS customer_status FROM customers ORDER BY clean_name ASC;"}
        ]
    },
    {
        "id": 2,
        "title": "Dynamic Substring Extraction & Delimiters",
        "question": "How do we dynamically parse customer first names and format custom badge codes?",
        "tables": ["customers"],
        "steps": [
            {"step": 1, "label": "INSTR()", "keyword": "INSTR(name, ' ')", "explanation": "Finds the 1-based numerical index of the whitespace character inside customer names.", "sql": "SELECT name, INSTR(name, ' ') AS delimiter_index FROM customers WHERE loyalty_level = 'Gold';"},
            {"step": 2, "label": "SUBSTR()", "keyword": "SUBSTR(name, 1, INSTR(name, ' ') - 1)", "explanation": "Slices the string starting at index 1 for the calculated character count.", "sql": "SELECT name, SUBSTR(name, 1, INSTR(name, ' ') - 1) AS first_name FROM customers WHERE loyalty_level = 'Gold';"},
            {"step": 3, "label": "|| (Concat)", "keyword": "'VIP-' || customer_id", "explanation": "Concatenates static text with customer ID coerced to text format.", "sql": "SELECT name, SUBSTR(name, 1, INSTR(name, ' ') - 1) AS first_name, 'VIP-' || customer_id AS vip_code FROM customers WHERE loyalty_level = 'Gold';"}
        ]
    },
    {
        "id": 3,
        "title": "Temporal Calculations & Julian Conversions",
        "question": "How many days have elapsed since each customer joined as of March 1, 2024?",
        "tables": ["customers"],
        "steps": [
            {"step": 1, "label": "STRFTIME()", "keyword": "STRFTIME('%Y', join_date)", "explanation": "Parses standard ISO date text to isolate the 4-digit year component.", "sql": "SELECT customer_id, name, join_date, STRFTIME('%Y', join_date) AS join_year FROM customers;"},
            {"step": 2, "label": "JULIANDAY()", "keyword": "JULIANDAY('2024-03-01') - JULIANDAY(join_date)", "explanation": "Converts calendar dates into continuous Julian day counts and subtracts them.", "sql": "SELECT name, join_date, (JULIANDAY('2024-03-01') - JULIANDAY(join_date)) AS raw_day_interval FROM customers;"},
            {"step": 3, "label": "ROUND()", "keyword": "ROUND(CAST(...) AS REAL, 0)", "explanation": "Casts calculation to real number and rounds off fractional days to integer values.", "sql": "SELECT customer_id, name, join_date, ROUND(CAST((JULIANDAY('2024-03-01') - JULIANDAY(join_date)) AS REAL), 0) AS days_as_member FROM customers ORDER BY days_as_member DESC;"}
        ]
    },
    {
        "id": 4,
        "title": "Relational Aggregations & Group Filters",
        "question": "What is the total revenue and order volume per city for cities generating over $15?",
        "tables": ["stores", "orders"],
        "steps": [
            {"step": 1, "label": "INNER JOIN", "keyword": "FROM stores s JOIN orders o ON s.store_id = o.store_id", "explanation": "Merges store locations with transactional orders using matching foreign keys.", "sql": "SELECT s.city, o.order_id, o.total_amount FROM stores s JOIN orders o ON s.store_id = o.store_id;"},
            {"step": 2, "label": "GROUP BY", "keyword": "GROUP BY s.city", "explanation": "Rolls individual records into summary groups per unique city.", "sql": "SELECT s.city, COUNT(o.order_id) AS total_orders, ROUND(SUM(o.total_amount), 2) AS total_revenue FROM stores s JOIN orders o ON s.store_id = o.store_id GROUP BY s.city;"},
            {"step": 3, "label": "HAVING", "keyword": "HAVING total_revenue > 15.00", "explanation": "Applies filtering criteria directly to computed group aggregate sums.", "sql": "SELECT s.city, COUNT(o.order_id) AS total_orders, ROUND(SUM(o.total_amount), 2) AS total_revenue FROM stores s JOIN orders o ON s.store_id = o.store_id GROUP BY s.city HAVING total_revenue > 15.00 ORDER BY total_revenue DESC;"}
        ]
    },
    {
        "id": 5,
        "title": "Set Operations (UNION vs UNION ALL)",
        "question": "How do set operators combine customer purchase activity across store branches?",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Branch 1", "keyword": "SELECT customer_id, 'Seattle' ... WHERE store_id = 1", "explanation": "Executes initial query retrieving store 1 transactions.", "sql": "SELECT customer_id, 'Seattle' AS store FROM orders WHERE store_id = 1;"},
            {"step": 2, "label": "UNION ALL", "keyword": "UNION ALL", "explanation": "Concatenates secondary result set directly, retaining duplicate instances.", "sql": "SELECT customer_id, 'Seattle' AS store FROM orders WHERE store_id = 1 UNION ALL SELECT customer_id, 'Portland' AS store FROM orders WHERE store_id = 2;"},
            {"step": 3, "label": "UNION", "keyword": "UNION (Deduplicated)", "explanation": "Performs set union operation, running implicit distinct sorting across both queries.", "sql": "SELECT customer_id FROM orders WHERE store_id = 1 UNION SELECT customer_id FROM orders WHERE store_id = 2;"}
        ]
    },
    {
        "id": 6,
        "title": "Cross-Tabulation Matrix Pivoting",
        "question": "How do we pivot unit sales into dedicated columns for Seattle and Portland?",
        "tables": ["products", "order_items", "orders"],
        "steps": [
            {"step": 1, "label": "LEFT JOIN", "keyword": "products LEFT JOIN order_items LEFT JOIN orders", "explanation": "Preserves all products even when no sales entries exist.", "sql": "SELECT p.name, oi.quantity, o.store_id FROM products p LEFT JOIN order_items oi ON p.product_id = oi.product_id LEFT JOIN orders o ON oi.order_id = o.order_id;"},
            {"step": 2, "label": "CASE in SUM", "keyword": "SUM(CASE WHEN store_id = 1 THEN quantity ELSE 0 END)", "explanation": "Conditionally maps units into store columns within aggregation calculations.", "sql": "SELECT p.name, SUM(CASE WHEN o.store_id = 1 THEN oi.quantity ELSE 0 END) AS seattle_units FROM products p LEFT JOIN order_items oi ON p.product_id = oi.product_id LEFT JOIN orders o ON oi.order_id = o.order_id GROUP BY p.product_id, p.name;"},
            {"step": 3, "label": "Pivoted View", "keyword": "Pivoted Multi-Column Aggregation", "explanation": "Completes matrix columns for each target location and summarizes overall units.", "sql": "SELECT p.name, p.category, SUM(CASE WHEN o.store_id = 1 THEN oi.quantity ELSE 0 END) AS seattle_units, SUM(CASE WHEN o.store_id = 2 THEN oi.quantity ELSE 0 END) AS portland_units, COALESCE(SUM(oi.quantity), 0) AS total_units_sold FROM products p LEFT JOIN order_items oi ON p.product_id = oi.product_id LEFT JOIN orders o ON oi.order_id = o.order_id GROUP BY p.product_id, p.name, p.category ORDER BY total_units_sold DESC;"}
        ]
    },
    {
        "id": 7,
        "title": "Window Ranking Schemes on Ties",
        "question": "How do ROW_NUMBER, RANK, and DENSE_RANK behave across identical prices in the Bakery category?",
        "tables": ["products"],
        "steps": [
            {"step": 1, "label": "ROW_NUMBER()", "keyword": "ROW_NUMBER() OVER (ORDER BY price DESC)", "explanation": "Arbitrarily splits ties by generating continuous incrementing integers (1, 2, 3, 4, 5).", "sql": "SELECT name, category, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS row_num FROM products WHERE category = 'Bakery';"},
            {"step": 2, "label": "RANK()", "keyword": "RANK() OVER (ORDER BY price DESC)", "explanation": "Detects identical $3.50 items, ranks them all as 3, and skips to rank 6 for the next value.", "sql": "SELECT name, category, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS row_num, RANK() OVER (PARTITION BY category ORDER BY price DESC) AS rank_with_gaps FROM products WHERE category = 'Bakery';"},
            {"step": 3, "label": "DENSE_RANK()", "keyword": "DENSE_RANK() OVER (ORDER BY price DESC)", "explanation": "Assigns rank 3 to all $3.50 items, but avoids skipping numbers for subsequent values.", "sql": "SELECT name, category, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS row_num, RANK() OVER (PARTITION BY category ORDER BY price DESC) AS rank_with_gaps, DENSE_RANK() OVER (PARTITION BY category ORDER BY price DESC) AS dense_rank_no_gaps FROM products WHERE category = 'Bakery';"}
        ]
    },
    {
        "id": 8,
        "title": "Deterministic Tie-Breaking & Secondary Keys",
        "question": "How do we eliminate nondeterministic orderings when values are identical?",
        "tables": ["products"],
        "steps": [
            {"step": 1, "label": "Nondeterministic Sort", "keyword": "ORDER BY price DESC", "explanation": "Sorting only by price causes ties ($4.75 Oat Milk vs Vanilla Latte) to shuffle randomly across runs.", "sql": "SELECT product_id, name, category, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS arbitrary_rn FROM products WHERE category = 'Espresso';"},
            {"step": 2, "label": "Deterministic Secondary Key", "keyword": "ORDER BY price DESC, product_id ASC", "explanation": "Adding unique product_id guarantees identical, repeatable row ranks in production pipelines.", "sql": "SELECT product_id, name, category, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC, product_id ASC) AS deterministic_rn FROM products WHERE category = 'Espresso';"}
        ]
    },
    {
        "id": 9,
        "title": "NULL Ordering Handling Across Ranking",
        "question": "How do we rank tiers while controlling whether NULLs appear first or last?",
        "tables": ["customers"],
        "steps": [
            {"step": 1, "label": "Default Engine Ordering", "keyword": "ORDER BY loyalty_level ASC", "explanation": "In SQLite and MySQL, NULL values sort to the top by default, polluting rank 1.", "sql": "SELECT name, loyalty_level, DENSE_RANK() OVER (ORDER BY loyalty_level ASC) AS default_rank FROM customers;"},
            {"step": 2, "label": "Explicit NULLS LAST Workaround", "keyword": "ORDER BY (loyalty_level IS NULL), loyalty_level ASC", "explanation": "Evaluates boolean (IS NULL) first (0 for valid data, 1 for NULL), forcing unassigned accounts to the bottom.", "sql": "SELECT name, loyalty_level, DENSE_RANK() OVER (ORDER BY (loyalty_level IS NULL) ASC, loyalty_level ASC) AS nulls_last_dense_rank FROM customers ORDER BY (loyalty_level IS NULL) ASC, loyalty_level ASC;"}
        ]
    },
    {
        "id": 10,
        "title": "Top-N Cutoffs with Ties (ROW_NUMBER vs DENSE_RANK)",
        "question": "How do we include all products tied for the top 2 price positions within each category?",
        "tables": ["products"],
        "steps": [
            {"step": 1, "label": "Lossy Cutoff (ROW_NUMBER)", "keyword": "WHERE rn <= 2", "explanation": "Arbitrarily cuts off tied items at row 2, hiding valid contenders.", "sql": "WITH ranked AS (SELECT category, name, price, ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC, product_id ASC) AS rn FROM products) SELECT category, name, price, rn FROM ranked WHERE rn <= 2;"},
            {"step": 2, "label": "Fair Cutoff (DENSE_RANK)", "keyword": "WHERE dense_rk <= 2", "explanation": "Ensures all items belonging to the top 2 price tiers are fully included.", "sql": "WITH ranked AS (SELECT category, name, price, DENSE_RANK() OVER (PARTITION BY category ORDER BY price DESC) AS dense_rk FROM products) SELECT category, name, price, dense_rk FROM ranked WHERE dense_rk <= 2 ORDER BY category, dense_rk;"}
        ]
    },
    {
        "id": 11,
        "title": "Statistical Distribution & Percentile Bucketing",
        "question": "How do NTILE, PERCENT_RANK, and CUME_DIST segment price distributions across the menu?",
        "tables": ["products"],
        "steps": [
            {"step": 1, "label": "NTILE(3) Bucketing", "keyword": "NTILE(3) OVER (ORDER BY price DESC)", "explanation": "Divides all products into 3 equal-sized tiers (High, Mid, Budget).", "sql": "SELECT name, price, NTILE(3) OVER (ORDER BY price DESC) AS price_tier_bucket FROM products;"},
            {"step": 2, "label": "PERCENT_RANK()", "keyword": "PERCENT_RANK() OVER (ORDER BY price ASC)", "explanation": "Computes relative rank ratio: (Rank - 1) / (Total Rows - 1).", "sql": "SELECT name, price, ROUND(PERCENT_RANK() OVER (ORDER BY price ASC), 2) AS relative_percent_rank FROM products;"},
            {"step": 3, "label": "CUME_DIST()", "keyword": "CUME_DIST() OVER (ORDER BY price ASC)", "explanation": "Calculates cumulative distribution percentage of rows with price <= current row.", "sql": "SELECT name, price, NTILE(3) OVER (ORDER BY price DESC) AS quartile, ROUND(PERCENT_RANK() OVER (ORDER BY price ASC), 2) AS pct_rank, ROUND(CUME_DIST() OVER (ORDER BY price ASC), 2) AS cume_dist FROM products;"}
        ]
    },
    {
        "id": 12,
        "title": "Window Offsets & Running Totals",
        "question": "How do we inspect previous order amounts and compute cumulative running revenue?",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "LAG()", "keyword": "LAG(total_amount, 1) OVER (ORDER BY order_date, order_id)", "explanation": "Fetches the total value from the previous row based on the global chronological order.", "sql": "SELECT order_id, order_date, total_amount, LAG(total_amount, 1) OVER (ORDER BY order_date, order_id) AS prev_amount FROM orders;"},
            {"step": 2, "label": "LEAD()", "keyword": "LEAD(total_amount, 1) OVER (ORDER BY order_date, order_id)", "explanation": "Peeks into the following transaction row to extract subsequent spending.", "sql": "SELECT order_id, order_date, total_amount, LAG(total_amount, 1) OVER (ORDER BY order_date, order_id) AS prev_amount, LEAD(total_amount, 1) OVER (ORDER BY order_date, order_id) AS next_amount FROM orders;"},
            {"step": 3, "label": "Running Total", "keyword": "SUM(total_amount) OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)", "explanation": "Computes running total sum from the first record up to the current evaluation row.", "sql": "SELECT order_id, order_date, total_amount, LAG(total_amount, 1) OVER (ORDER BY order_date, order_id) AS prev_amount, SUM(total_amount) OVER (ORDER BY order_date, order_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total FROM orders;"}
        ]
    },
    {
        "id": 13,
        "title": "Nested Scalar & List Subqueries",
        "question": "Which customers have placed orders larger than the chain-wide average spend?",
        "tables": ["customers", "orders"],
        "steps": [
            {"step": 1, "label": "Scalar Subquery", "keyword": "(SELECT AVG(total_amount) FROM orders)", "explanation": "Computes overall chain average order spend as an isolated scalar float value.", "sql": "SELECT ROUND(AVG(total_amount), 2) AS chain_avg_order FROM orders;"},
            {"step": 2, "label": "Inner Filter", "keyword": "WHERE total_amount > (SELECT AVG(...))", "explanation": "Filters order records against the scalar average and outputs matching customer IDs.", "sql": "SELECT DISTINCT customer_id FROM orders WHERE total_amount > (SELECT AVG(total_amount) FROM orders);"},
            {"step": 3, "label": "Outer Filter", "keyword": "WHERE customer_id IN (...)", "explanation": "Filters customers based on membership in the inner subquery result set.", "sql": "SELECT name, loyalty_level FROM customers WHERE customer_id IN (SELECT customer_id FROM orders WHERE total_amount > (SELECT AVG(total_amount) FROM orders));"}
        ]
    },
    {
        "id": 14,
        "title": "Modular Chained CTEs",
        "question": "Which customers have average spending that exceeds chain-wide benchmarks?",
        "tables": ["customers", "orders"],
        "steps": [
            {"step": 1, "label": "CTE 1 (Benchmark)", "keyword": "WITH global_stats AS (SELECT AVG(total_amount) ...)", "explanation": "Stores chain average spend inside an isolated, named common table expression.", "sql": "WITH global_stats AS (SELECT AVG(total_amount) AS global_avg FROM orders) SELECT * FROM global_stats;"},
            {"step": 2, "label": "CTE 2 (Aggregation)", "keyword": "customer_summary AS (SELECT c.name, AVG(o.total_amount) ...)", "explanation": "Chains a secondary CTE to aggregate totals and order sizes per customer.", "sql": "WITH customer_summary AS (SELECT c.name, SUM(o.total_amount) AS total_spent, AVG(o.total_amount) AS cust_avg FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.name) SELECT * FROM customer_summary;"},
            {"step": 3, "label": "CROSS JOIN Filter", "keyword": "FROM customer_summary cs CROSS JOIN global_stats gs WHERE ...", "explanation": "Combines customer aggregates with benchmark scalars to filter higher-spending accounts.", "sql": "WITH global_stats AS (SELECT AVG(total_amount) AS global_avg FROM orders), customer_summary AS (SELECT c.name, SUM(o.total_amount) AS total_spent, AVG(o.total_amount) AS cust_avg FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.name) SELECT cs.name, ROUND(cs.total_spent, 2) AS total_spent, ROUND(cs.cust_avg, 2) AS cust_avg, ROUND(gs.global_avg, 2) AS benchmark FROM customer_summary cs CROSS JOIN global_stats gs WHERE cs.cust_avg > gs.global_avg;"}
        ]
    },
    {
        "id": 15,
        "title": "Recursive CTE: Series Generation",
        "question": "How can we dynamically construct a 5-level loyalty threshold table recursively?",
        "tables": [],
        "steps": [
            {"step": 1, "label": "Anchor Query", "keyword": "SELECT 1 AS tier_level, 10.00 AS spend_threshold, 'Free Upsize'", "explanation": "Initializes the recursive stack with the starting seed row.", "sql": "SELECT 1 AS tier_level, 10.00 AS spend_threshold, 'Free Upsize' AS perk;"},
            {"step": 2, "label": "Recursive UNION", "keyword": "UNION ALL SELECT tier_level + 1 ... WHERE tier_level < 5", "explanation": "Iterates on the growing table by adding subsequent rows until the WHERE limit is met.", "sql": "WITH RECURSIVE reward_tiers(tier_level, spend_threshold, perk) AS (SELECT 1, 10.00, 'Free Upsize' UNION ALL SELECT tier_level + 1, spend_threshold + 15.00, 'Free Pastry' FROM reward_tiers WHERE tier_level < 5) SELECT * FROM reward_tiers;"}
        ]
    },
    {
        "id": 16,
        "title": "Recursive CTE: Organizational Hierarchy Walk",
        "question": "How do we traverse the reporting chain from the CEO down to all store baristas?",
        "tables": ["employees"],
        "steps": [
            {"step": 1, "label": "Root Anchor", "keyword": "WHERE manager_id IS NULL", "explanation": "Extracts the top-level tree root node where manager_id is NULL.", "sql": "SELECT employee_id, name, title, manager_id, 1 AS depth_level, name AS path FROM employees WHERE manager_id IS NULL;"},
            {"step": 2, "label": "Tree Traversal", "keyword": "JOIN org_chart o ON e.manager_id = o.employee_id", "explanation": "Recursively joins direct reports to managers, tracking tree depth and hierarchy paths.", "sql": "WITH RECURSIVE org_chart AS (SELECT employee_id, name, title, manager_id, 1 AS depth_level, name AS path FROM employees WHERE manager_id IS NULL UNION ALL SELECT e.employee_id, e.name, e.title, e.manager_id, o.depth_level + 1, o.path || ' -> ' || e.name FROM employees e JOIN org_chart o ON e.manager_id = o.employee_id) SELECT depth_level, name, title, path FROM org_chart ORDER BY depth_level, name;"}
        ]
    },
    {
        "id": 17,
        "title": "Semi-Structured JSON Parsing",
        "question": "How do we extract nested JSON equipment logs per store?",
        "tables": ["store_logs", "stores"],
        "steps": [
            {"step": 1, "label": "Root JSON Key", "keyword": "JSON_EXTRACT(log_payload, '$.shift')", "explanation": "Extracts the value of root-level scalar key 'shift' from JSON strings.", "sql": "SELECT store_id, JSON_EXTRACT(log_payload, '$.shift') AS shift_name FROM store_logs;"},
            {"step": 2, "label": "Nested JSON Path", "keyword": "JSON_EXTRACT(log_payload, '$.equip_status.espresso_machine')", "explanation": "Uses dot-notation to extract child object properties inside JSON logs.", "sql": "SELECT s.store_name, JSON_EXTRACT(l.log_payload, '$.shift') AS shift_name, JSON_EXTRACT(l.log_payload, '$.equip_status.espresso_machine') AS espresso_status FROM store_logs l JOIN stores s ON l.store_id = s.store_id;"}
        ]
    },
    {
        "id": 18,
        "title": "Anti-Joins for Inactive Entity Discovery",
        "question": "Which customers registered but have never placed an order at any store location?",
        "tables": ["customers", "orders"],
        "steps": [
            {"step": 1, "label": "LEFT JOIN", "keyword": "customers c LEFT JOIN orders o ON c.customer_id = o.customer_id", "explanation": "Preserves every customer record, populating order columns with NULL when unlinked.", "sql": "SELECT c.customer_id, c.name, o.order_id FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id;"},
            {"step": 2, "label": "WHERE IS NULL", "keyword": "WHERE o.order_id IS NULL", "explanation": "Filters for unmatched right-table keys to isolate dormant customers.", "sql": "SELECT c.customer_id, c.name, c.join_date FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id WHERE o.order_id IS NULL ORDER BY c.join_date ASC;"}
        ]
    },
    {
        "id": 19,
        "title": "Window Boundaries: FIRST_VALUE & LAST_VALUE",
        "question": "What was the first item added versus the priciest item in each order?",
        "tables": ["order_items", "products"],
        "steps": [
            {"step": 1, "label": "FIRST_VALUE()", "keyword": "FIRST_VALUE(p.name) OVER (PARTITION BY oi.order_id ORDER BY oi.item_id)", "explanation": "Retrieves the product value of the first record entered inside the order partition.", "sql": "SELECT oi.order_id, p.name, oi.item_price, FIRST_VALUE(p.name) OVER (PARTITION BY oi.order_id ORDER BY oi.item_id ASC) AS first_item FROM order_items oi JOIN products p ON oi.product_id = p.product_id;"},
            {"step": 2, "label": "LAST_VALUE() Frame", "keyword": "ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING", "explanation": "Expands window boundaries so LAST_VALUE evaluates across the entire partition.", "sql": "SELECT oi.order_id, p.name, oi.item_price, FIRST_VALUE(p.name) OVER (PARTITION BY oi.order_id ORDER BY oi.item_id ASC) AS first_item, LAST_VALUE(p.name) OVER (PARTITION BY oi.order_id ORDER BY oi.item_price ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS priciest_item FROM order_items oi JOIN products p ON oi.product_id = p.product_id;"}
        ]
    },
    {
        "id": 20,
        "title": "Sliding Frame Moving Averages",
        "question": "What is the 3-order moving average of transaction values across the chain?",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Base Window Sort", "keyword": "OVER (ORDER BY order_date, order_id)", "explanation": "Establishes a sorted order timeline for moving aggregations.", "sql": "SELECT order_id, order_date, total_amount FROM orders ORDER BY order_date, order_id;"},
            {"step": 2, "label": "Sliding Frame", "keyword": "ROWS BETWEEN 2 PRECEDING AND CURRENT ROW", "explanation": "Restricts calculation window to the current row and preceding 2 transactions.", "sql": "SELECT order_id, order_date, total_amount, ROUND(AVG(total_amount) OVER (ORDER BY order_date, order_id ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3_orders FROM orders;"}
        ]
    },
    {
        "id": 21,
        "title": "JSON Filter Predicates",
        "question": "Which stores have espresso machines that need service based on JSON logs?",
        "tables": ["store_logs", "stores"],
        "steps": [
            {"step": 1, "label": "JSON Extraction", "keyword": "JSON_EXTRACT(log_payload, '$.equip_status.espresso_machine')", "explanation": "Pulls machine operational status from nested JSON attributes.", "sql": "SELECT s.store_name, s.city, JSON_EXTRACT(l.log_payload, '$.equip_status.espresso_machine') AS machine_status FROM store_logs l JOIN stores s ON l.store_id = s.store_id;"},
            {"step": 2, "label": "JSON WHERE Filter", "keyword": "WHERE JSON_EXTRACT(...) != 'OK'", "explanation": "Directly filters rows based on the value parsed out of JSON payload.", "sql": "SELECT s.store_name, s.city, JSON_EXTRACT(l.log_payload, '$.equip_status.espresso_machine') AS machine_status FROM store_logs l JOIN stores s ON l.store_id = s.store_id WHERE JSON_EXTRACT(l.log_payload, '$.equip_status.espresso_machine') != 'OK';"}
        ]
    },
    {
        "id": 22,
        "title": "Retention Cadence: Partitioned Lag & Date Differencing",
        "question": "How many days elapse between repeat purchases for returning customers?",
        "tables": ["orders", "customers"],
        "steps": [
            {"step": 1, "label": "Partitioned LAG()", "keyword": "LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)", "explanation": "Extracts previous order dates for each customer across their purchase history.", "sql": "SELECT c.name, o.order_date, LAG(o.order_date) OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS prev_date FROM orders o JOIN customers c ON o.customer_id = c.customer_id;"},
            {"step": 2, "label": "Date Difference", "keyword": "JULIANDAY(order_date) - JULIANDAY(LAG(...))", "explanation": "Computes whole days between repeat visits to calculate purchase intervals.", "sql": "SELECT c.name, o.order_date, LAG(o.order_date) OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS prev_date, ROUND(CAST(JULIANDAY(o.order_date) - JULIANDAY(LAG(o.order_date) OVER (PARTITION BY c.customer_id ORDER BY o.order_date)) AS REAL), 0) AS days_between_orders FROM orders o JOIN customers c ON o.customer_id = c.customer_id;"}
        ]
    },
    {
        "id": 23,
        "title": "Google Level: Gaps & Islands (Consecutive Daily Activity)",
        "question": "Find all customers who placed orders on consecutive calendar days.",
        "tables": ["orders", "customers"],
        "steps": [
            {"step": 1, "label": "Deduplicate Dates", "keyword": "SELECT DISTINCT customer_id, order_date", "explanation": "Isolates unique customer activity dates to prevent multiple purchases on the same day from breaking contiguous streak calculations.", "sql": "SELECT DISTINCT customer_id, order_date FROM orders ORDER BY customer_id, order_date;"},
            {"step": 2, "label": "ROW_NUMBER Offset", "keyword": "ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date)", "explanation": "Calculates an ordinal ranking integer. In Gaps & Islands, subtracting the row rank from the calendar day creates an invariant group key for consecutive sequences.", "sql": "SELECT customer_id, order_date, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn FROM (SELECT DISTINCT customer_id, order_date FROM orders);"},
            {"step": 3, "label": "Island Identification", "keyword": "DATE(order_date, '-' || rn || ' days') AS island_group", "explanation": "Computes the anchor date. Consecutive dates advance at the exact same rate as the row number, yielding an identical island date key for all continuous rows in a streak.", "sql": "WITH distinct_dates AS (SELECT DISTINCT customer_id, order_date FROM orders), ranked AS (SELECT customer_id, order_date, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn FROM distinct_dates) SELECT customer_id, order_date, DATE(order_date, '-' || rn || ' days') AS streak_id FROM ranked;"},
            {"step": 4, "label": "Streak Filter", "keyword": "HAVING COUNT(*) >= 2", "explanation": "Groups records by customer and the computed streak anchor, filtering exclusively for streaks that persisted for 2 or more consecutive days.", "sql": "WITH distinct_dates AS (SELECT DISTINCT customer_id, order_date FROM orders), ranked AS (SELECT customer_id, order_date, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn FROM distinct_dates), islands AS (SELECT customer_id, order_date, DATE(order_date, '-' || rn || ' days') AS streak_id FROM ranked) SELECT c.name, COUNT(*) AS consecutive_days, MIN(i.order_date) AS streak_start, MAX(i.order_date) AS streak_end FROM islands i JOIN customers c ON i.customer_id = c.customer_id GROUP BY i.customer_id, i.streak_id HAVING COUNT(*) >= 2;"}
        ]
    },
    {
        "id": 24,
        "title": "Google Level: Sessionization via Inactivity Thresholds",
        "question": "Group user purchases into distinct sessions separated by an inactivity gap of 2 or more days.",
        "tables": ["orders", "customers"],
        "steps": [
            {"step": 1, "label": "LAG Interval", "keyword": "LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)", "explanation": "Extracts the previous transaction timestamp to compute the elapsed duration between events for each individual user.", "sql": "SELECT order_id, customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date FROM orders;"},
            {"step": 2, "label": "Session Boundary Flag", "keyword": "CASE WHEN (JULIANDAY(order_date) - JULIANDAY(prev_date)) >= 2 THEN 1 ELSE 0 END", "explanation": "Evaluates time delta. Emits a 1 whenever elapsed time exceeds the session threshold (2 days) or for the user's initial purchase; emits 0 otherwise.", "sql": "WITH gaps AS (SELECT order_id, customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date FROM orders) SELECT order_id, customer_id, order_date, CASE WHEN prev_date IS NULL OR (JULIANDAY(order_date) - JULIANDAY(prev_date)) >= 2 THEN 1 ELSE 0 END AS is_new_session FROM gaps;"},
            {"step": 3, "label": "Cumulative Session ID", "keyword": "SUM(is_new_session) OVER (PARTITION BY customer_id ORDER BY order_date)", "explanation": "Computes a running sum of boundary flags. Because 0 does not change the running sum, transactions belonging to the same continuous window inherit identical session numbers.", "sql": "WITH gaps AS (SELECT order_id, customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date FROM orders), flagged AS (SELECT order_id, customer_id, order_date, CASE WHEN prev_date IS NULL OR (JULIANDAY(order_date) - JULIANDAY(prev_date)) >= 2 THEN 1 ELSE 0 END AS is_new_session FROM gaps) SELECT c.name, f.order_id, f.order_date, SUM(f.is_new_session) OVER (PARTITION BY f.customer_id ORDER BY f.order_date) AS user_session_id FROM flagged f JOIN customers c ON f.customer_id = c.customer_id;"}
        ]
    },
    {
        "id": 25,
        "title": "Google Level: Market Basket Co-Occurrence (Self-Join)",
        "question": "Which distinct pairs of products are most frequently purchased together within the same order?",
        "tables": ["order_items", "products"],
        "steps": [
            {"step": 1, "label": "Cartesian Non-Equi Join", "keyword": "FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id", "explanation": "Self-joins order items on matching order IDs. The non-equi predicate a.product_id < b.product_id prevents pairing an item with itself and suppresses reverse duplicates.", "sql": "SELECT a.order_id, a.product_id AS item_1, b.product_id AS item_2 FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id;"},
            {"step": 2, "label": "Product Metadata Joins", "keyword": "JOIN products p1 ON a.product_id = p1.product_id ...", "explanation": "Attaches human-readable product names to both sides of the paired relation.", "sql": "SELECT a.order_id, p1.name AS product_a, p2.name AS product_b FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id JOIN products p1 ON a.product_id = p1.product_id JOIN products p2 ON b.product_id = p2.product_id;"},
            {"step": 3, "label": "Frequency Rank", "keyword": "GROUP BY product_a, product_b ORDER BY pair_frequency DESC", "explanation": "Counts occurrences of each unique product pair across orders to determine affinity ranking.", "sql": "SELECT p1.name AS product_a, p2.name AS product_b, COUNT(*) AS pair_frequency FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id JOIN products p1 ON a.product_id = p1.product_id JOIN products p2 ON b.product_id = p2.product_id GROUP BY p1.name, p2.name ORDER BY pair_frequency DESC, product_a ASC;"}
        ]
    },
    {
        "id": 26,
        "title": "Google Level: Pareto Principle (Cumulative Revenue Distribution)",
        "question": "Calculate the running percentage of total company revenue generated by each customer.",
        "tables": ["orders", "customers"],
        "steps": [
            {"step": 1, "label": "Customer Aggregate", "keyword": "GROUP BY customer_id ORDER BY customer_spend DESC", "explanation": "Sums total spend per customer and sorts accounts from highest contributor to lowest.", "sql": "SELECT customer_id, ROUND(SUM(total_amount), 2) AS customer_spend FROM orders GROUP BY customer_id ORDER BY customer_spend DESC;"},
            {"step": 2, "label": "Window Cumulative Sum", "keyword": "SUM(customer_spend) OVER (ORDER BY customer_spend DESC)", "explanation": "Calculates the running cumulative spend total across customers sorted by highest contribution.", "sql": "WITH cust_totals AS (SELECT customer_id, SUM(total_amount) AS customer_spend FROM orders GROUP BY customer_id) SELECT customer_id, customer_spend, SUM(customer_spend) OVER (ORDER BY customer_spend DESC) AS running_spend FROM cust_totals;"},
            {"step": 3, "label": "Pareto Share Ratio", "keyword": "ROUND((running_spend / SUM(customer_spend) OVER ()) * 100.0, 2)", "explanation": "Divides the cumulative running total by an unbounded window sum across the entire dataset to yield running percentage share.", "sql": "WITH cust_totals AS (SELECT customer_id, SUM(total_amount) AS customer_spend FROM orders GROUP BY customer_id), running AS (SELECT c.name, ct.customer_spend, SUM(ct.customer_spend) OVER (ORDER BY ct.customer_spend DESC) AS running_spend, SUM(ct.customer_spend) OVER () AS total_company_revenue FROM cust_totals ct JOIN customers c ON ct.customer_id = c.customer_id) SELECT name, ROUND(customer_spend, 2) AS customer_spend, ROUND(running_spend, 2) AS running_spend, ROUND((running_spend / total_company_revenue) * 100.0, 2) AS cumulative_pct_share FROM running ORDER BY customer_spend DESC;"}
        ]
    },
    {
        "id": 27,
        "title": "Google Level: Peak Concurrency & Overlapping Shift Intervals",
        "question": "What is the peak number of staff working simultaneously at Store 1 on Feb 1, 2024?",
        "tables": ["store_shifts"],
        "steps": [
            {"step": 1, "label": "Timeline Normalization", "keyword": "UNION ALL (+1 for Start, -1 for End)", "explanation": "Unrolls start and end timestamps into discrete atomic events. Starting a shift adds +1 to concurrency; ending a shift subtracts -1.", "sql": "SELECT start_time AS event_time, 1 AS delta FROM store_shifts WHERE store_id = 1 UNION ALL SELECT end_time AS event_time, -1 AS delta FROM store_shifts WHERE store_id = 1 ORDER BY event_time ASC;"},
            {"step": 2, "label": "Running Headcount", "keyword": "SUM(delta) OVER (ORDER BY event_time ASC ROWS UNBOUNDED PRECEDING)", "explanation": "Executes a chronological running total over the +/-1 flags to track active staff count at every timestamp change.", "sql": "WITH events AS (SELECT start_time AS event_time, 1 AS delta FROM store_shifts WHERE store_id = 1 UNION ALL SELECT end_time AS event_time, -1 AS delta FROM store_shifts WHERE store_id = 1) SELECT event_time, delta, SUM(delta) OVER (ORDER BY event_time ASC, delta DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS active_headcount FROM events;"},
            {"step": 3, "label": "Peak Extraction", "keyword": "MAX(active_headcount)", "explanation": "Scans calculated running headcounts to locate maximum simultaneous concurrency across the store shift logs.", "sql": "WITH events AS (SELECT start_time AS event_time, 1 AS delta FROM store_shifts WHERE store_id = 1 UNION ALL SELECT end_time AS event_time, -1 AS delta FROM store_shifts WHERE store_id = 1), concurrency AS (SELECT event_time, SUM(delta) OVER (ORDER BY event_time ASC, delta DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS active_headcount FROM events) SELECT MAX(active_headcount) AS peak_simultaneous_staff FROM concurrency;"}
        ]
    },
    {
        "id": 28,
        "title": "Google Level: Full User MoM Retention Lifecycle",
        "question": "Categorize every customer in February 2024 as New, Retained, or Churned relative to January.",
        "tables": ["orders", "customers"],
        "steps": [
            {"step": 1, "label": "Monthly Cohort Maps", "keyword": "DISTINCT customer_id, STRFTIME('%Y-%m', order_date)", "explanation": "Builds discrete boolean activity sets for January 2024 and February 2024.", "sql": "SELECT DISTINCT customer_id, STRFTIME('%Y-%m', order_date) AS active_month FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-02-29' ORDER BY customer_id;"},
            {"step": 2, "label": "Cross-Month Joining", "keyword": "LEFT JOIN jan_cohort vs feb_cohort", "explanation": "Pairs customer presence between January and February to analyze transitions.", "sql": "WITH jan AS (SELECT DISTINCT customer_id FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-01-31'), feb AS (SELECT DISTINCT customer_id FROM orders WHERE order_date BETWEEN '2024-02-01' AND '2024-02-29') SELECT c.customer_id, c.name, CASE WHEN j.customer_id IS NOT NULL THEN 1 ELSE 0 END AS in_jan, CASE WHEN f.customer_id IS NOT NULL THEN 1 ELSE 0 END AS in_feb FROM customers c LEFT JOIN jan j ON c.customer_id = j.customer_id LEFT JOIN feb f ON c.customer_id = f.customer_id WHERE j.customer_id IS NOT NULL OR f.customer_id IS NOT NULL;"},
            {"step": 3, "label": "Lifecycle Classification", "keyword": "CASE WHEN in_jan = 1 AND in_feb = 1 THEN 'Retained' ...", "explanation": "Evaluates transition truth tables to classify accounts as Retained, Churned, or Newly Acquired.", "sql": "WITH jan AS (SELECT DISTINCT customer_id FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-01-31'), feb AS (SELECT DISTINCT customer_id FROM orders WHERE order_date BETWEEN '2024-02-01' AND '2024-02-29') SELECT c.name, CASE WHEN j.customer_id IS NOT NULL AND f.customer_id IS NOT NULL THEN 'Retained Active' WHEN j.customer_id IS NOT NULL AND f.customer_id IS NULL THEN 'Churned from Jan' WHEN j.customer_id IS NULL AND f.customer_id IS NOT NULL THEN 'New / Acquired' END AS feb_lifecycle_status FROM customers c LEFT JOIN jan j ON c.customer_id = j.customer_id LEFT JOIN feb f ON c.customer_id = f.customer_id WHERE j.customer_id IS NOT NULL OR f.customer_id IS NOT NULL ORDER BY feb_lifecycle_status, c.name;"}
        ]
    },
    {
        "id": 29,
        "title": "Google Level: Time-Series Imputation & Forward Fill",
        "question": "Impute missing calendar days for Store 1 and forward-fill total sales across Feb 1 to Feb 5.",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Recursive Calendar Grid", "keyword": "WITH RECURSIVE calendar(day_date) AS (...)", "explanation": "Generates a synthetic dense continuous date spine to bridge calendar gaps.", "sql": "WITH RECURSIVE calendar(dt) AS (SELECT '2024-02-01' UNION ALL SELECT DATE(dt, '+1 day') FROM calendar WHERE dt < '2024-02-05') SELECT dt AS calendar_date FROM calendar;"},
            {"step": 2, "label": "Dense Left Join", "keyword": "calendar LEFT JOIN orders ON calendar_date = order_date", "explanation": "Joins store daily totals against calendar spine, revealing null sales on inactive dates.", "sql": "WITH RECURSIVE calendar(dt) AS (SELECT '2024-02-01' UNION ALL SELECT DATE(dt, '+1 day') FROM calendar WHERE dt < '2024-02-05'), store_daily AS (SELECT order_date, SUM(total_amount) AS daily_rev FROM orders WHERE store_id = 1 GROUP BY order_date) SELECT c.dt, s.daily_rev FROM calendar c LEFT JOIN store_daily s ON c.dt = s.order_date ORDER BY c.dt;"},
            {"step": 3, "label": "Forward-Fill Imputation", "keyword": "COALESCE(daily_rev, (Subquery Last Known Non-Null))", "explanation": "Replaces NULL day values with the most recent prior non-null sales observation.", "sql": "WITH RECURSIVE calendar(dt) AS (SELECT '2024-02-01' UNION ALL SELECT DATE(dt, '+1 day') FROM calendar WHERE dt < '2024-02-05'), store_daily AS (SELECT order_date, SUM(total_amount) AS daily_rev FROM orders WHERE store_id = 1 GROUP BY order_date), merged AS (SELECT c.dt, s.daily_rev FROM calendar c LEFT JOIN store_daily s ON c.dt = s.order_date) SELECT m.dt AS sales_date, m.daily_rev AS raw_recorded_rev, COALESCE(m.daily_rev, (SELECT m2.daily_rev FROM merged m2 WHERE m2.dt < m.dt AND m2.daily_rev IS NOT NULL ORDER BY m2.dt DESC LIMIT 1)) AS forward_filled_sales FROM merged m ORDER BY m.dt;"}
        ]
    },
    {
        "id": 30,
        "title": "Google Level: Period-Over-Period Growth Rates (MoM)",
        "question": "What is the month-over-month revenue growth rate for the entire chain between Jan and Feb 2024?",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Monthly Aggregation", "keyword": "STRFTIME('%Y-%m', order_date) ... SUM(total_amount)", "explanation": "Rolls orders into distinct calendar month summary buckets.", "sql": "SELECT STRFTIME('%Y-%m', order_date) AS rev_month, ROUND(SUM(total_amount), 2) AS monthly_sales FROM orders GROUP BY rev_month ORDER BY rev_month;"},
            {"step": 2, "label": "LAG Offset", "keyword": "LAG(monthly_sales, 1) OVER (ORDER BY rev_month)", "explanation": "Extracts previous month sales figure to align metrics side-by-side.", "sql": "WITH monthly AS (SELECT STRFTIME('%Y-%m', order_date) AS rev_month, ROUND(SUM(total_amount), 2) AS monthly_sales FROM orders GROUP BY rev_month) SELECT rev_month, monthly_sales, LAG(monthly_sales, 1) OVER (ORDER BY rev_month) AS prior_month_sales FROM monthly;"},
            {"step": 3, "label": "Percentage Growth", "keyword": "ROUND(((curr - prior) / prior) * 100.0, 2)", "explanation": "Calculates relative month-over-month delta percentage.", "sql": "WITH monthly AS (SELECT STRFTIME('%Y-%m', order_date) AS rev_month, ROUND(SUM(total_amount), 2) AS monthly_sales FROM orders GROUP BY rev_month), comp AS (SELECT rev_month, monthly_sales, LAG(monthly_sales, 1) OVER (ORDER BY rev_month) AS prior_month_sales FROM monthly) SELECT rev_month, monthly_sales, prior_month_sales, ROUND(((monthly_sales - prior_month_sales) / prior_month_sales) * 100.0, 2) AS mom_growth_pct FROM comp WHERE prior_month_sales IS NOT NULL;"}
        ]
    },
    {
        "id": 31,
        "title": "Google Level: Non-Aggregate Median & Percentile Bucketing",
        "question": "Find the true median order amount across all orders without using a built-in MEDIAN function.",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Dual-Direction Ranking", "keyword": "ROW_NUMBER() ASC vs ROW_NUMBER() DESC", "explanation": "Ranks transactions in both directions. The median rows satisfy ABS(rank_asc - rank_desc) <= 1.", "sql": "SELECT order_id, total_amount, ROW_NUMBER() OVER (ORDER BY total_amount ASC, order_id ASC) AS rk_asc, ROW_NUMBER() OVER (ORDER BY total_amount DESC, order_id DESC) AS rk_desc FROM orders;"},
            {"step": 2, "label": "Median Element Filter", "keyword": "WHERE rk_asc BETWEEN rk_desc - 1 AND rk_desc + 1", "explanation": "Isolates the single center element (for odd row counts) or dual center elements (for even row counts).", "sql": "WITH ranked AS (SELECT order_id, total_amount, ROW_NUMBER() OVER (ORDER BY total_amount ASC, order_id ASC) AS rk_asc, ROW_NUMBER() OVER (ORDER BY total_amount DESC, order_id DESC) AS rk_desc FROM orders) SELECT order_id, total_amount, rk_asc, rk_desc FROM ranked WHERE rk_asc BETWEEN rk_desc - 1 AND rk_desc + 1;"},
            {"step": 3, "label": "Median Average", "keyword": "ROUND(AVG(total_amount), 2)", "explanation": "Averages the isolated center values to output the exact mathematical median.", "sql": "WITH ranked AS (SELECT order_id, total_amount, ROW_NUMBER() OVER (ORDER BY total_amount ASC, order_id ASC) AS rk_asc, ROW_NUMBER() OVER (ORDER BY total_amount DESC, order_id DESC) AS rk_desc FROM orders) SELECT ROUND(AVG(total_amount), 2) AS calculated_median_spend FROM ranked WHERE rk_asc BETWEEN rk_desc - 1 AND rk_desc + 1;"}
        ]
    },
    {
        "id": 32,
        "title": "Google Level: Dynamic Sliding Calendar Windows (Rolling 3-Day Revenue)",
        "question": "Calculate the true rolling 3-day revenue for every order date (current day + 2 previous days).",
        "tables": ["orders"],
        "steps": [
            {"step": 1, "label": "Daily Aggregation", "keyword": "GROUP BY order_date", "explanation": "Collapses transactional volume down to distinct daily revenue totals.", "sql": "SELECT order_date, ROUND(SUM(total_amount), 2) AS daily_rev FROM orders GROUP BY order_date ORDER BY order_date;"},
            {"step": 2, "label": "Self Non-Equi Date Join", "keyword": "ON a.order_date >= b.order_date AND b.order_date >= DATE(a.order_date, '-2 days')", "explanation": "Pairs each target calendar day with all observed active dates falling within its trailing 3-day window.", "sql": "WITH daily AS (SELECT order_date, SUM(total_amount) AS daily_rev FROM orders GROUP BY order_date) SELECT a.order_date AS target_date, b.order_date AS matched_prior_date, b.daily_rev FROM daily a JOIN daily b ON b.order_date BETWEEN DATE(a.order_date, '-2 days') AND a.order_date ORDER BY target_date, matched_prior_date;"},
            {"step": 3, "label": "Rolling Window Sum", "keyword": "SUM(b.daily_rev) GROUP BY a.order_date", "explanation": "Sums revenue across the non-equi join to compute a date-accurate rolling 3-day metric.", "sql": "WITH daily AS (SELECT order_date, SUM(total_amount) AS daily_rev FROM orders GROUP BY order_date) SELECT a.order_date, ROUND(SUM(b.daily_rev), 2) AS rolling_3day_spend, COUNT(b.order_date) AS active_days_in_window FROM daily a JOIN daily b ON b.order_date BETWEEN DATE(a.order_date, '-2 days') AND a.order_date GROUP BY a.order_date ORDER BY a.order_date ASC;"}
        ]
    }
]

In [4]:
import subprocess, time, sys
from flask import Flask, jsonify, render_template_string

app = Flask(__name__)

# ==============================================================================
# BLOCK 4: EMBEDDED FRONTEND INTERFACE (HTML, CSS & VANILLA JS)
# Purpose: Single-page application layout that runs entirely inside the client
#          browser, polling Flask endpoints without requiring page reloads.
# ==============================================================================
HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>INTERACTIVE SQL PLAYBOOK</title>
    <!-- Prism.js CSS for syntax highlighting -->
    <link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
    <style>
        :root {
            --bg: #f8fafc;
            --surface: #ffffff;
            --border: #e2e8f0;
            --text-main: #0f172a;
            --text-muted: #64748b;
            --primary: #2563eb;
            --primary-light: #eff6ff;
            --accent: #059669;
            --code-bg: #f1f5f9;
        }
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background-color: var(--bg); color: var(--text-main); margin: 0; padding: 24px; }
        .container { max-width: 1100px; margin: 0 auto; }
        .header-card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 20px 24px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px; }
        .nav-controls { display: flex; align-items: center; gap: 10px; }
        select { padding: 8px 12px; border: 1px solid var(--border); border-radius: 6px; font-size: 14px; background: white; color: var(--text-main); max-width: 440px; }
        .btn { background: var(--primary); color: white; border: none; padding: 8px 16px; border-radius: 6px; cursor: pointer; font-weight: 500; font-size: 14px; }
        .btn:disabled { background: #cbd5e1; cursor: not-allowed; }
        .main-card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 24px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); margin-bottom: 20px; }
        .title { margin-top: 0; color: var(--text-main); font-size: 20px; }
        .question-box {
            background: var(--primary-light);
            border-left: 4px solid var(--primary);
            padding: 12px 16px;
            border-radius: 4px;
            margin-bottom: 20px;
            font-size: 18px; /* Larger font size */
            color: var(--primary); /* Brighter color */
            font-weight: 700; /* Bolder font */
            /* Removed: text-shadow: 0 0 5px rgba(37, 99, 235, 0.5); /* Neon effect */
        }
        .steps-nav { display: flex; gap: 8px; margin-bottom: 16px; flex-wrap: wrap; }
        .step-btn { background: var(--code-bg); border: 1px solid var(--border); padding: 8px 14px; border-radius: 6px; cursor: pointer; font-size: 13px; font-weight: 600; color: var(--text-muted); transition: all 0.15s; }
        .step-btn.active { background: var(--primary); color: white; border-color: var(--primary); }
        .step-card { background: #ffffff; border: 1px solid var(--border); border-radius: 8px; padding: 16px; margin-bottom: 20px; }
        .keyword-badge { display: inline-block; background: #e0e7ff; color: #3730a3; padding: 4px 8px; border-radius: 4px; font-family: monospace; font-weight: 600; font-size: 13px; margin-bottom: 8px; }
        .explanation-text { margin: 8px 0 12px 0; font-size: 14px; color: #334155; line-height: 1.5; }
        .sql-box { background: var(--code-bg); border: 1px solid var(--border); border-radius: 6px; overflow: hidden; }
        .table-section-title { font-size: 14px; font-weight: 600; color: var(--text-muted); margin: 16px 0 8px 0; display: flex; align-items: center; gap: 6px; }
        .table-responsive { overflow-x: auto; border: 1px solid var(--border); border-radius: 6px; }
        table { width: 100%; border-collapse: collapse; font-size: 13px; text-align: left; background: white; }
        th { background: #f8fafc; color: #475569; font-weight: 600; padding: 10px 12px; border-bottom: 1px solid var(--border); }
        td { padding: 8px 12px; border-bottom: 1px solid var(--border); color: #1e293b; }
        tr:last-child td { border-bottom: none; }

        /* Modal Styles */
        .modal {
            display: none;
            position: fixed;
            z-index: 1000;
            left: 0;
            top: 0;
            width: 100%;
            height: 100%;
            overflow: auto;
            background-color: rgba(0,0,0,0.4);
            backdrop-filter: blur(3px);
        }
        .modal-content {
            background-color: var(--surface);
            margin: 10% auto;
            padding: 20px;
            border: 1px solid var(--border);
            border-radius: 12px;
            width: 90%;
            max-width: 900px;
            box-shadow: 0 4px 8px 0 rgba(0,0,0,0.2), 0 6px 20px 0 rgba(0,0,0,0.19);
            animation-name: animatetop;
            animation-duration: 0.4s;
            position: relative;
        }
        @keyframes animatetop {
            from {top: -300px; opacity: 0}
            to {top: 0; opacity: 1}
        }
        .modal-close-btn {
            color: #aaa;
            float: right;
            font-size: 28px;
            font-weight: bold;
            cursor: pointer;
        }
        .modal-close-btn:hover,
        .modal-close-btn:focus {
            color: black;
            text-decoration: none;
            cursor: pointer;
        }
        .modal-body-content {
            max-height: 70vh;
            overflow-y: auto;
            padding-top: 10px;
        }
        .modal-table-section {
            margin-bottom: 20px;
        }
        .modal-table-section h4 {
            margin-top: 0;
            margin-bottom: 10px;
            color: var(--primary);
        }

        /* Prism.js code container overrides */
        pre[class*="language-"] {
            margin: 0;
            padding: 14px 18px;
            border-radius: 6px;
            background: var(--code-bg);
            border: none;
            font-size: 13px;
            line-height: 1.6;
            white-space: pre-wrap !important;
            word-break: break-word !important;
            overflow-x: hidden !important;
            box-sizing: border-box;
        }
        code[class*="language-"] {
            white-space: pre-wrap !important;
            word-break: break-word !important;
        }
        .token.comment, .token.prolog, .token.doctype, .token.cdata {
            color: slategray;
        }
    </style>
</head>
<body>
<div class="container">
    <div class="header-card">
        <div>
            <span style="font-size: 13px; font-weight: 600; color: var(--accent);">INTERACTIVE SQL PLAYBOOK</span>
            <h2 style="margin: 4px 0 0 0; font-size: 18px;">33 Scenarios (Foundations to Google/FAANG Level)</h2>
        </div>
        <div class="nav-controls">
            <button id="prevBtn" class="btn" onclick="prevScenario()">Previous</button>
            <select id="scenarioSelect" onchange="switchScenario(this.value)">
                {% for sc in scenarios %}
                    <option value="{{ sc.id }}">Scenario {{ sc.id + 1 }}: {{ sc.title }}</option>
                {% endfor %}
            </select>
            <button id="nextBtn" class="btn" onclick="nextScenario()">Next</button>
        </div>
    </div>
    <div class="main-card">
        <h3 id="scenarioTitle" class="title"></h3>
        <div id="scenarioQuestion" class="question-box"></div>

        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px;">
            <div class="table-section-title">INTERACTIVE STEP-BY-STEP KEYWORD EXECUTION</div>
            <button id="viewSourceTablesBtn" class="btn" onclick="displaySourceTablesInModal()" style="background: #64748b; font-size: 12px; padding: 6px 12px;">View Source Tables</button>
        </div>

        <div id="stepsNav" class="steps-nav"></div>
        <div class="step-card">
            <div><span id="keywordBadge" class="keyword-badge"></span></div>
            <div id="stepExplanation" class="explanation-text"></div>
            <!-- SQL Code Block with Prism.js -->
            <div class="sql-box"><pre><code id="stepSqlCode" class="language-sql"></code></pre></div>
        </div>
        <div class="table-section-title">RESULTING OUTPUT MATRIX (AT THIS STEP)</div>
        <div id="tableOutput" class="table-responsive"></div>
    </div>
</div>

<!-- The Modal -->
<div id="sourceTablesModal" class="modal">
    <div class="modal-content">
        <span class="modal-close-btn" onclick="hideSourceTablesModal()">&times;</span>
        <h3>Source Tables for this Scenario</h3>
        <div id="modalTablesContent" class="modal-body-content"></div>
    </div>
</div>

<!-- Prism.js core and SQL language component -->
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
    let currentScenarioId = 0;
    let currentStepIndex = 0;
    let scenarioData = null;

    function getScenarioCount() {
        const select = document.getElementById('scenarioSelect');
        return select ? select.options.length : 33;
    }

    function loadScenario(id) {
        fetch('/api/scenario/' + id)
            .then(res => {
                if (!res.ok) throw new Error('Network response was not ok');
                return res.json();
            })
            .then(data => {
                scenarioData = data;
                currentScenarioId = data.id;
                currentStepIndex = 0;
                document.getElementById('scenarioSelect').value = currentScenarioId;
                document.getElementById('scenarioTitle').innerText = 'Scenario ' + (data.id + 1) + ': ' + data.title;
                document.getElementById('scenarioQuestion').innerText = data.question;

                const nav = document.getElementById('stepsNav');
                nav.innerHTML = '';
                data.steps.forEach((step, idx) => {
                    const btn = document.createElement('button');
                    btn.className = 'step-btn ' + (idx === 0 ? 'active' : '');
                    btn.id = 'step-btn-' + idx;
                    btn.innerText = 'Step ' + step.step + ': ' + step.label;
                    btn.onclick = () => renderStep(idx);
                    nav.appendChild(btn);
                });
                renderStep(0);
                updateControls();
                updateSourceTablesButton();
            })
            .catch(err => console.error("Failed to load scenario:", err));
    }

    function renderStep(idx) {
        currentStepIndex = idx;
        document.querySelectorAll('.step-btn').forEach((b, i) => {
            b.classList.toggle('active', i === idx);
        });
        const step = scenarioData.steps[idx];
        document.getElementById('keywordBadge').innerText = step.keyword;
        document.getElementById('stepExplanation').innerText = step.explanation;
        document.getElementById('stepSqlCode').textContent = step.sql;
        Prism.highlightElement(document.getElementById('stepSqlCode'));
        document.getElementById('tableOutput').innerHTML = step.table_html;
    }

    function updateControls() {
        const total = getScenarioCount();
        document.getElementById('prevBtn').disabled = (currentScenarioId <= 0);
        document.getElementById('nextBtn').disabled = (currentScenarioId >= total - 1);
    }

    function switchScenario(id) { loadScenario(parseInt(id, 10)); }
    function prevScenario() { if (currentScenarioId > 0) loadScenario(currentScenarioId - 1); }
    function nextScenario() {
        const total = getScenarioCount();
        if (currentScenarioId < total - 1) loadScenario(currentScenarioId + 1);
    }

    // Modal Functions
    function showSourceTablesModal() {
        document.getElementById('sourceTablesModal').style.display = 'block';
    }

    function hideSourceTablesModal() {
        document.getElementById('sourceTablesModal').style.display = 'none';
    }

    function displaySourceTablesInModal() {
        const modalContentDiv = document.getElementById('modalTablesContent');
        modalContentDiv.innerHTML = '';

        if (scenarioData && scenarioData.source_tables_html) {
            if (Object.keys(scenarioData.source_tables_html).length === 0) {
                modalContentDiv.innerHTML = '<p>No specific source tables are directly involved in this scenario, or data could not be fetched.</p>';
            } else {
                for (const tableName in scenarioData.source_tables_html) {
                    const tableSection = document.createElement('div');
                    tableSection.className = 'modal-table-section';
                    tableSection.innerHTML = `<h4>Table: ${tableName}</h4><div class="table-responsive">${scenarioData.source_tables_html[tableName]}</div>`;
                    modalContentDiv.appendChild(tableSection);
                }
            }
        } else {
            modalContentDiv.innerHTML = '<p>Source table data not available for this scenario.</p>';
        }
        showSourceTablesModal();
    }

    function updateSourceTablesButton() {
        const btn = document.getElementById('viewSourceTablesBtn');
        if (scenarioData && scenarioData.tables && scenarioData.tables.length > 0) {
            btn.style.display = 'inline-block';
        } else {
            btn.style.display = 'none';
        }
    }

    window.onclick = function(event) {
        const modal = document.getElementById('sourceTablesModal');
        if (event.target == modal) {
            modal.style.display = 'none';
        }
    }

    window.addEventListener('DOMContentLoaded', () => { loadScenario(0); });
</script>
</body>
</html>
"""

In [5]:
# ==============================================================================
# BLOCK 5: FLASK ROUTE HANDLERS & QUERY EXECUTION ENGINE
# Purpose: Flask application logic to serve the HTML and execute SQL queries
#          using pandas.read_sql_query, with automatic multiline SQL formatting.
# ==============================================================================
import re

app = Flask(__name__)

def format_sql_multiline(query: str) -> str:
    """
    Transforms single-line SQL queries into clean, readable multiline formatting
    by structuring major clauses and parenthetical blocks.
    """
    clauses = [
        "WITH RECURSIVE", "WITH",
        "SELECT DISTINCT", "SELECT",
        "FROM", "LEFT JOIN", "INNER JOIN", "CROSS JOIN", "JOIN",
        "WHERE", "GROUP BY", "HAVING",
        "ORDER BY", "LIMIT",
        "UNION ALL", "UNION"
    ]

    formatted = query.strip()

    # Break into newlines before major top-level SQL clauses
    for clause in clauses:
        pattern = rf'\b({clause})\b'
        formatted = re.sub(pattern, r'\n\1', formatted, flags=re.IGNORECASE)

    # Format and indent lines based on scope
    raw_lines = [line.strip() for line in formatted.split('\n') if line.strip()]
    indent_level = 0
    structured_lines = []

    for line in raw_lines:
        # Step out of indent for closing parenthesis
        if line.startswith(')') and indent_level > 0:
            indent_level -= 1

        structured_lines.append(('    ' * indent_level) + line)

        # Step into indent when a nested query / CTE begins
        if line.endswith('('):
            indent_level += 1

    return '\n'.join(structured_lines)


@app.route('/')
def index():
    """Renders the single-page HTML layout with the scenario dropdown pre-populated."""
    return render_template_string(HTML_TEMPLATE, scenarios=SCENARIOS)


@app.route('/api/scenario/<int:scenario_id>')
def get_scenario(scenario_id):
    """
    Executes all SQL queries corresponding to the requested scenario in real time,
    formats outputs using pandas.DataFrame.to_html, auto-formats SQL queries,
    and delivers payload via JSON.
    """
    if scenario_id < 0 or scenario_id >= len(SCENARIOS):
        return jsonify({"error": "Scenario not found"}), 404

    sc = SCENARIOS[scenario_id]
    steps_with_results = []
    source_tables_data = {}

    # Fetch and format source tables
    for table_name in sc['tables']:
        try:
            df_source = pd.read_sql_query(f"SELECT * FROM {table_name}", db_conn)
            source_tables_data[table_name] = df_source.to_html(index=False, classes='matrix-table')
        except Exception as e:
            source_tables_data[table_name] = f"<div style='padding: 10px; color: red;'>Error fetching table '{table_name}': {str(e)}</div>"

    # Execute query steps and apply multiline formatting for display
    for s in sc['steps']:
        try:
            df = pd.read_sql_query(s['sql'], db_conn)
            table_html = df.to_html(index=False, classes='matrix-table')
        except Exception as e:
            table_html = f"<div style='padding: 10px; color: red;'>Query Execution Error: {str(e)}</div>"

        steps_with_results.append({
            "step": s["step"],
            "label": s["label"],
            "keyword": s["keyword"],
            "explanation": s["explanation"],
            "sql": format_sql_multiline(s["sql"]),
            "table_html": table_html
        })

    return jsonify({
        "id": sc["id"],
        "title": sc["title"],
        "question": sc["question"],
        "tables": sc["tables"],
        "steps": steps_with_results,
        "source_tables_html": source_tables_data
    })

In [6]:
# ==============================================================================
# BLOCK 6: BACKGROUND THREAD WORKER & LOCALTUNNEL BRIDGE
# Purpose: Boots Flask, gets the external IP password, and creates an instant
#          HTTPS tunnel using localtunnel.
# ==============================================================================
import threading
import urllib.request
import time
import subprocess

print("[3/5] Starting Flask app in background thread...")
server_thread = threading.Thread(
    target=lambda: app.run(port=5000, host="0.0.0.0", use_reloader=False, debug=False),
    daemon=True
)
server_thread.start()

# Health check: Wait until Flask answers on port 5000
for _ in range(15):
    try:
        with urllib.request.urlopen("http://127.0.0.1:5000/", timeout=1) as res:
            if res.status == 200:
                break
    except Exception:
        time.sleep(0.5)

# Fetch public endpoint IP (localtunnel asks for this as an access password once)
try:
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except Exception:
    endpoint_ip = "Unavailable"

print("[4/5] Establishing secure public tunnel via Localtunnel...")
# Kill any previous tunnel instances
subprocess.run(["pkill", "-f", "lt"], stderr=subprocess.DEVNULL)

# Start localtunnel in background
lt_process = subprocess.Popen(
    ["npx", "-y", "localtunnel", "--port", "5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Read the generated URL from localtunnel output
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 15:
    line = lt_process.stdout.readline()
    if "your url is:" in line.lower():
        tunnel_url = line.split()[-1].strip()
        break

print("\n" + "="*70)
print("🚀 INTERACTIVE SQL PLAYBOOK IS READY!")
if tunnel_url:
    print(f"\n👉 1. Click this link: {tunnel_url}")
    print(f"👉 2. If prompted for 'Tunnel Password', paste this IP: {endpoint_ip}")
else:
    print("⚠️ Tunnel initialization timed out. Please re-run this cell.")
print("="*70)

[3/5] Starting Flask app in background thread...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Sep/2026 10:24:57] "GET / HTTP/1.1" 200 -


[4/5] Establishing secure public tunnel via Localtunnel...

🚀 INTERACTIVE SQL PLAYBOOK IS READY!

👉 1. Click this link: https://rotten-lions-worry.loca.lt
👉 2. If prompted for 'Tunnel Password', paste this IP: 34.24.209.49
